In [2]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. До,авляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь до,авлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\WildPositionMonitor
✅ Sys path: ['c:\\Users\\123\\Desktop\\WildPositionMonitor', 'C:\\Users\\123\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\123\\AppData\\Local\\Programs\\Python\\Python312\\DLLs']...


In [9]:
from src_new.core.database import Database

from sqlalchemy import text 

class Psql:
    """Класс для подключения к PostgreSQL и выполнения запросов."""
    def __init__(self, engine=None):
        if engine:
            self.engine = engine
        else:
            self.engine = Database.get_engine() 

    def get_our_articles(self):
        """Получаем список наших артикулов из БД."""
        query = text("SELECT DISTINCT article_id FROM card_data")
        df = Database.read_sql_to_dataframe(query)
        return df['article_id'].tolist()

In [ ]:
# Получаем список артикулов для проверки из БД
product_list = Psql().get_our_articles()

[34943247,
 61426830,
 61426831,
 61470353,
 61769118,
 62103844,
 62103845,
 62103846,
 62103847,
 62111591,
 62111594,
 62123562,
 62664789,
 62664790,
 62668786,
 62673150,
 62693191,
 62904918,
 63094114,
 63101972,
 63252128,
 63710536,
 63714804,
 64359676,
 64359677,
 64359678,
 64400407,
 64409719,
 64490285,
 64490286,
 64490287,
 64515489,
 64520318,
 64522260,
 64523377,
 64533542,
 64533543,
 64533544,
 64533818,
 64536456,
 64536457,
 64536458,
 65250850,
 65387751,
 65417076,
 65417077,
 65417078,
 65417185,
 65420083,
 65420084,
 65691784,
 65691786,
 65837723,
 65837725,
 70042724,
 72521524,
 72521525,
 72521527,
 72521529,
 73165638,
 73173308,
 74691662,
 74943679,
 77969335,
 77969336,
 77969337,
 77969338,
 77969339,
 77969340,
 80001964,
 80001966,
 85320488,
 86812140,
 86812142,
 86812143,
 92133605,
 92133606,
 92133607,
 92133608,
 92142020,
 92142024,
 92142025,
 92142026,
 92142027,
 92142028,
 93105465,
 94139441,
 94479875,
 94479876,
 94479877,
 94757810,

In [2]:
import asyncio
import json
import os
import random
from curl_cffi.requests import AsyncSession

# Импортируем настройки из нашего конфигурационного файла
from src_new.config import (
    MAX_ARTICLES_TO_PROCESS,
    PRODUCT_LIST,
    WB_PRICE_URL,
    WB_RAW_COOKIES,
)


class WbPriceParser:
    # URL для получения информации о цене товара
    url = WB_PRICE_URL
    # Куки для прохождения проверок безопасности Wildberries
    cookies = WB_RAW_COOKIES

    @staticmethod
    async def get_wb_product_data(session, item_id):
        """Асинхронный запрос к внутреннему API Wildberries для одного товара."""
        params = {"appType": 1, "curr": "rub", "dest": "-1257786", "nm": item_id}

        headers = {
            "Accept": "*/*",
            "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8",
            "Connection": "keep-alive",
            "Cookie": WbPriceParser.cookies,
            "Origin": "https://www.wildberries.ru",
            "Referer": f"https://www.wildberries.ru/catalog/{item_id}/detail.aspx",
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/147.0.0.0 Safari/537.36"
            ),
        }

        try:
            response = await session.get(
                WbPriceParser.url, params=params, headers=headers, timeout=15
            )

            if response.status_code == 200:
                return item_id, response.json()

            return item_id, {
                "error": f"Сервер вернул код ошибки: {response.status_code}"
            }

        except Exception as e:
            return item_id, {"error": f"Ошибка сети при запросе: {str(e)}"}

    @staticmethod
    async def process_product(session, item_id, semaphore, index, total):
        """Обработка одного товара внутри батча с ограничением параллельных запросов."""
        async with semaphore:
            print(
                f"   [Товар {index}/{total}] Получение данных для артикула: {item_id}"
            )

            item_id, result = await WbPriceParser.get_wb_product_data(
                session, item_id
            )

            if "error" not in result:
                print(f"      ✅ Данные для {item_id} успешно получены!")
            else:
                print(f"      ❌ Ошибка для {item_id}: {result['error']}")

            # Случайная пауза между запросами от 0.5 до 1.5 секунд, чтобы имитировать действия человека
            await asyncio.sleep(random.uniform(0.5, 1.5))

            return item_id, result

    @staticmethod
    async def parse_batch(session, batch_products, max_concurrent=10):
        """Асинхронно обрабатывает ОДИН конкретный батч (порцию) товаров."""
        results = {}
        # Семафор ограничивает количество одновременно выполняемых запросов внутри батча
        semaphore = asyncio.Semaphore(max_concurrent)

        tasks = [
            WbPriceParser.process_product(
                session=session,
                item_id=item_id,
                semaphore=semaphore,
                index=index,
                total=len(batch_products),
            )
            for index, item_id in enumerate(batch_products, 1)
        ]

        # Запускаем все задачи текущего батча параллельно и ждем их выполнения
        responses = await asyncio.gather(*tasks)

        for item_id, data in responses:
            results[item_id] = data

        return results

    @staticmethod
    def save_results(new_data, filename="wb_products_data.json"):
        """Дозаписывает новые данные в JSON-файл, сохраняя старые."""
        existing_data = {}

        # Если файл уже существует, сначала считываем из него старые данные
        if os.path.exists(filename):
            try:
                with open(filename, "r", encoding="utf-8") as f:
                    existing_data = json.load(f)
            except Exception as e:
                print(
                    f"⚠️ Не удалось прочитать существующий файл {filename} ({e}), создаем новый."
                )

        # Объединяем старые данные с новыми результатами
        # Если артикул уже был, его данные обновятся на самые свежие
        existing_data.update(new_data)

        # Записываем объединенные данные обратно в файл
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(existing_data, f, indent=4, ensure_ascii=False)

        print(f"💾 Данные успешно сохранены в файл {filename}. Всего товаров в базе: {len(existing_data)}")


def get_batches(target_list, batch_size=50):
    """Вспомогательная функция (генератор), которая режет список на куски по batch_size."""
    for i in range(0, len(target_list), batch_size):
        yield target_list[i : i + batch_size]


async def main():
    print("🚀 Запуск парсера Wildberries...")

    # 1. Берем исходный список артикулов
    all_products = PRODUCT_LIST

    # 2. Проверяем ограничение на количество из config.py (для тестов)
    if MAX_ARTICLES_TO_PROCESS is not None:
        print(f"⚙️ Включено ограничение тестов: обрабатываем только первые {MAX_ARTICLES_TO_PROCESS} артикулов.")
        all_products = all_products[:MAX_ARTICLES_TO_PROCESS]

    total_products = len(all_products)
    batch_size = 50  # Размер одной порции
    
    print(f"📦 Всего к обработке: {total_products} артикулов. Размер батча: {batch_size}")

    # Создаем одну общую сессию для всех запросов (это экономит ресурсы компьютера)
    async with AsyncSession(impersonate="chrome110") as session:
        
        # Делим список на батчи и перебираем их по очереди
        # enumerate(..., 1) нужен, чтобы красиво выводить номера батчей (Батч 1, Батч 2 и т.д.)
        for batch_index, batch in enumerate(get_batches(all_products, batch_size), 1):
            print(f"\n🔹 Обработка пакета (батча) №{batch_index} | Товаров в пакете: {len(batch)}")
            
            # Запускаем парсинг текущего батча
            batch_results = await WbPriceParser.parse_all_products_by_batch_dummy(session, batch)
            
            # Как только батч завершился — сразу сохраняем его результаты в файл
            WbPriceParser.save_results(batch_results)
            
            # Небольшая пауза между батчами (3–5 секунд), чтобы дать отдохнуть API Wildberries
            if (batch_index * batch_size) < total_products:
                sleep_time = random.uniform(3.0, 5.0)
                print(f"⏳ Батч №{batch_index} завершен. Спим {sleep_time:.1f} сек. перед следующим батчем...")
                await asyncio.sleep(sleep_time)

    print("\n🎉 Программа успешно завершила работу! Все данные сохранены.")

In [3]:
# Метод-прослойка для совместимости, вызывает логику обработки одного батча
WbPriceParser.parse_all_products_by_batch_dummy = WbPriceParser.parse_batch

await main()

🚀 Запуск парсера Wildberries...
⚙️ Включено ограничение тестов: обрабатываем только первые 100 артикулов.
📦 Всего к обработке: 59 артикулов. Размер батча: 50

🔹 Обработка пакета (батча) №1 | Товаров в пакете: 50
   [Товар 1/50] Получение данных для артикула: 189954586
   [Товар 2/50] Получение данных для артикула: 489570061
   [Товар 3/50] Получение данных для артикула: 489570482
   [Товар 4/50] Получение данных для артикула: 489570201
   [Товар 5/50] Получение данных для артикула: 489570288
   [Товар 6/50] Получение данных для артикула: 209667566
   [Товар 7/50] Получение данных для артикула: 189579546
   [Товар 8/50] Получение данных для артикула: 209666680
   [Товар 9/50] Получение данных для артикула: 209664375
   [Товар 10/50] Получение данных для артикула: 489569911
      ✅ Данные для 489570061 успешно получены!
      ✅ Данные для 189579546 успешно получены!
      ✅ Данные для 209664375 успешно получены!
      ✅ Данные для 489569911 успешно получены!
      ✅ Данные для 209666680 

In [13]:
import json

with open("wb_products_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [92]:
def price_data(data):
    arts_with_price = []
    for art in data:
        # Определяем перечень артикулов с данными по товарам
        products = data[art].get("products", None)
        for p in products:
            sizes = p.get("sizes", None)
            price = sizes[0].get("price", None)
            if price:
                arts_with_price.append({int(art): price.get("product", None)/100})
    return arts_with_price

prices = price_data(data) 

In [93]:
prices

[{189954586: 336.0},
 {489570482: 1674.0},
 {489570481: 2154.0},
 {160163535: 393.0},
 {154790374: 493.0}]